In [1]:
"""
task11_mc_binary_pairs.py — MC Binary Pairs (Wang et al., AAAI 2025)
=====================================================================
Track:       Metacognition
Benchmark:   MSV Metacognition Benchmark

OVERVIEW
--------
Full MC = d*/d-hat computation via type-2 Signal Detection Theory
(Maniscalco & Lau, 2012), adapted for LLMs by Wang et al. (2025)
as LLM-Meta-SDT. This is an SDT-inspired approximation of metacognitive
efficiency following the psychophysics literature.

PROTOCOL
--------
For each GPQA question, the model receives TWO binary judgment trials:

    Trial A (signal-present): "Is [CORRECT_ANSWER] the correct answer?"
    Trial B (signal-absent):  "Is [DISTRACTOR] the correct answer?"

The model responds yes/no with confidence 1-4 to each trial.

From the N*2 responses, we compute:
    hit_rate  = P(said yes | correct answer shown)
    fa_rate   = P(said yes | distractor shown)
    d-hat     = norm_ppf(hit_rate) - norm_ppf(fa_rate)
                (object-level sensitivity)

    type-2 AUC = P(conf on correct judgment > conf on incorrect judgment)
                 (meta-sensitivity, Maniscalco & Lau 2012)
    d*        = sqrt(2) * norm_ppf(type2_AUC)

    MC = d* / d-hat   (metacognitive efficiency ratio)

MC INTERPRETATION
-----------------
    MC ~ 1.0:  optimal — meta-sensitivity matches object-level sensitivity
    MC < 1.0:  metacognitive inefficiency — meta-sensitivity < object-level
    MC > 1.0:  unusual / potentially artifactual — needs careful interpretation
    MC = INDETERMINATE when d-hat ~ 0 (chance accuracy, expected on GPQA)

PER-TASK RETURN: normalized MC score = clip((MC + 1) / 2, 0, 1)
    Maps MC=-1 -> 0.0, MC=0 -> 0.5, MC=1 -> 1.0.
    INDETERMINATE returns 0.5 (neutral, flagged in analysis).

NOTE: This task constructs binary pairs directly from the GPQA Diamond
dataset — no separate JSONL file needed. For each question, it uses the
correct answer as the signal trial and a randomly selected incorrect
option as the distractor trial.

COST: 2 prompts per question * 80 questions = 160 prompts per model.

"""

import kaggle_benchmarks as kbench
import json, re, os, math, random
import pandas as pd



def _safe_prompt(llm, text):
    """Call llm.prompt() with graceful error handling.
    Returns response string, or None on any API/model failure.
    Logs failures for debugging but does not crash the task."""
    try:
        resp = llm.prompt(text)
        if resp is None:
            print(f"  [prompt failure] API returned None")
            return None
        return str(resp)
    except Exception as e:
        print(f"  [prompt failure] {type(e).__name__}: {e}")
        return None

def _norm_ppf(p):
    """Inverse normal CDF. Rational approximation, no scipy needed."""
    p = max(0.001, min(0.999, p))
    if p < 0.5:
        return -_norm_ppf(1.0 - p)
    t = math.sqrt(-2.0 * math.log(1.0 - p))
    c = (2.515517, 0.802853, 0.010328)
    d = (1.432788, 0.189269, 0.001308)
    return t - (c[0] + c[1]*t + c[2]*t*t) / (1 + d[0]*t + d[1]*t*t + d[2]*t*t*t)


def _parse_binary_judgment(response):
    """Parse binary yes/no judgment with confidence. Returns (judgment, conf).

    Returns ("UNKNOWN", None) if parsing fails — these are excluded from
    SDT computation to avoid biasing hit/FA rates.
    """
    text = str(response).strip()
    all_matches = re.findall(r'\{[^{}]*\}', text)
    for jm in reversed(all_matches):
        try:
            d = json.loads(jm)
            j = str(d.get("judgment", "")).lower().strip()
            if j in ("yes", "true", "correct", "1"):
                judgment = "yes"
            elif j in ("no", "false", "incorrect", "0"):
                judgment = "no"
            else:
                continue
            conf = d.get("confidence", 2)
            conf = 2 if conf is None else max(1, min(4, int(conf)))
            return judgment, conf
        except:
            continue
    # Keyword fallback (explicit yes/no only)
    if re.search(r'\byes\b', text, re.I):
        return "yes", 2
    if re.search(r'\bno\b', text, re.I):
        return "no", 2
    # No clear signal — return UNKNOWN to avoid biasing SDT computation
    return "UNKNOWN", None


# ── Task Definition ───────────────────────────────────────────────────────────
"""Full MC = d*/d-hat computation (Wang et al., AAAI 2025).

    Implements Maniscalco & Lau (2012) type-2 SDT framework adapted for
    LLMs. Each question generates two binary judgment trials (signal +
    noise). SDT statistics computed from aggregate hit/FA rates and
    type-2 AUC from confidence-weighted judgment correctness.

    Args:
        llm: Kaggle-injected model proxy.
        questions_json: JSON array of objects, each with keys:
            question, correct_answer_text, distractor_text

    Returns:
        float: Normalized MC score (0.0-1.0). 0.5 = indeterminate.
"""
@kbench.task(name="t11-msv_mc_binary_pairs", description="MC binary pairs: Wang-style type-2 SDT computation of MC = d*/d-hat from binary judgments.")
def mc_binary_pairs(llm) -> float:
    """Task 11: Full MC = d*/d-hat computation (Wang et al., AAAI 2025).

    Constructs binary pairs from GPQA Diamond data, runs all signal+noise
    trials, computes SDT statistics, saves CSV, returns normalized MC.
    """
    DATA_DIR = "/kaggle/input/msv-benchmark-data"
    questions = pd.read_csv(os.path.join(DATA_DIR, "gpqa_sampled_200.csv"))
    candidates = pd.read_csv(os.path.join(DATA_DIR, "gpqa_kaggle_candidates.csv"))
    task_df = questions.merge(candidates[["question_id", "difficulty"]], on="question_id", how="inner")

    option_cols = ["option_a", "option_b", "option_c", "option_d"]
    letter_map = {"A": "option_a", "B": "option_b", "C": "option_c", "D": "option_d"}
    random.seed(42)

    # SDT accumulators
    hits, misses, fas, crs = [], [], [], []
    conf_correct_judgment, conf_incorrect_judgment = [], []
    trial_rows = []
    n_parse_failures = 0

    print(f"Task 11 - MC Binary Pairs: {len(task_df)} questions ({len(task_df)*2} total prompts)")

    _overflow = False
    for _, row in task_df.iterrows():
        if _overflow:
            break
        correct_letter = row.correct_answer.strip().upper()
        correct_col = letter_map.get(correct_letter, "option_a")
        correct_text = str(row[correct_col])
        distractor_cols = [c for c in option_cols if c != correct_col]
        distractor_col = random.choice(distractor_cols)
        distractor_text = str(row[distractor_col])

        for is_signal, candidate in [(True, correct_text), (False, distractor_text)]:
            prompt = (
                "You will be shown a question and a candidate answer. "
                "Judge ONLY whether the candidate is the correct answer. "
                "Do not try to solve the question yourself.\n"
                "Rate your confidence that the candidate is correct.\n\n"
                "Question: " + row.question + "\n\n"
                "Candidate answer: " + candidate + "\n\n"
                "Is this the correct answer? Respond with ONLY JSON:\n"
                '{"judgment": "yes", "confidence": 3}\n'
                "or\n"
                '{"judgment": "no", "confidence": 2}\n'
                "YOUR RESPONSE MUST BE ONLY JSON. NO OTHER TEXT."
            )
            response = _safe_prompt(llm, prompt)
            if response is None:
                _overflow = True
                print(f'  Prompt failure (see error above) — returning partial results')
                break
            judgment, conf = _parse_binary_judgment(response)

            # Skip UNKNOWN judgments to avoid biasing SDT computation
            if judgment == "UNKNOWN":
                n_parse_failures += 1
                trial_rows.append({"question_id": row.question_id, "is_signal": is_signal,
                                   "judgment": "UNKNOWN", "confidence": None,
                                   "judgment_correct": None,
                                   "raw_response": (response or "")[:500]})
                continue

            said_yes = (judgment == "yes")

            if is_signal:
                (hits if said_yes else misses).append(conf)
            else:
                (fas if said_yes else crs).append(conf)

            judgment_correct = (is_signal and said_yes) or (not is_signal and not said_yes)
            if judgment_correct:
                conf_correct_judgment.append(conf)
            else:
                conf_incorrect_judgment.append(conf)

            trial_rows.append({"question_id": row.question_id, "is_signal": is_signal,
                               "said_yes": said_yes, "confidence": conf,
                               "judgment_correct": judgment_correct,
                               "raw_response": str(response)[:500]})

    # Save per-trial CSV
    pd.DataFrame(trial_rows).to_csv("/output/t11_mc_binary_pairs_results.csv", index=False)

    # SDT computation
    n_signal = len(hits) + len(misses)
    n_noise = len(fas) + len(crs)
    n_total_trials = n_signal + n_noise + n_parse_failures
    completion_rate = n_total_trials / (len(task_df) * 2) if len(task_df) > 0 else 0.0

    print(f"  Valid trials: {n_signal + n_noise}/{n_total_trials} | "
          f"Parse failures: {n_parse_failures} | "
          f"Completion: {n_total_trials}/{len(task_df)*2} ({completion_rate:.0%})")

    if n_signal == 0 or n_noise == 0:
        print("  No valid trials collected — returning 0.5")
        return round(0.5 * completion_rate, 4)

    hit_rate = (len(hits) + 0.5) / (n_signal + 1.0)
    fa_rate = (len(fas) + 0.5) / (n_noise + 1.0)
    d_hat = _norm_ppf(hit_rate) - _norm_ppf(fa_rate)

    print(f"  Hit rate: {hit_rate:.3f} | FA rate: {fa_rate:.3f} | d-hat: {d_hat:.3f}")

    if abs(d_hat) < 0.05:
        print("  d-hat near zero — INDETERMINATE (expected on GPQA Diamond)")
        # Save aggregate summary even for indeterminate cases
        sdt_summary = {"hit_rate": hit_rate, "fa_rate": fa_rate, "d_hat": d_hat,
                       "type2_auc": None, "d_star": None, "mc_raw": None,
                       "mc_normalized": 0.5, "indeterminate": True,
                       "n_parse_failures": n_parse_failures}
        with open("/output/t11_sdt_summary.json", "w") as f:
            json.dump(sdt_summary, f, indent=2)
        return round(0.5 * completion_rate, 4)

    n_type2 = 0
    wins = 0
    for c_conf in conf_correct_judgment:
        for i_conf in conf_incorrect_judgment:
            n_type2 += 1
            if c_conf > i_conf:
                wins += 1
            elif c_conf == i_conf:
                wins += 0.5

    if n_type2 == 0:
        return round(0.5 * completion_rate, 4)

    type2_auc = max(0.001, min(0.999, wins / n_type2))
    d_star = math.sqrt(2.0) * _norm_ppf(type2_auc)
    mc = d_star / d_hat
    mc_clipped = max(-1.0, min(1.0, mc))
    normalized = round((mc_clipped + 1.0) / 2.0, 4)

    # MC interpretation (Maniscalco & Lau, 2012):
    #   MC ~ 1.0: meta-sensitivity matches object-level sensitivity (efficient)
    #   MC < 1.0: metacognitive inefficiency (meta-sensitivity < object-level)
    #   MC > 1.0: unusual / potentially artifactual (needs careful interpretation)
    #   MC indeterminate when d-hat ~ 0 (no object-level ability to metacognate over)
    mc_label = ("efficient" if abs(mc - 1.0) < 0.2
                else "inefficient" if mc < 1.0 else "unusually high")
    print(f"  Type-2 AUC: {type2_auc:.3f} | d*: {d_star:.3f} | "
          f"MC: {mc:.3f} ({mc_label}) | Normalized: {normalized}")

    # Save aggregate SDT summary for offline analysis
    sdt_summary = {"hit_rate": round(hit_rate, 4), "fa_rate": round(fa_rate, 4),
                   "d_hat": round(d_hat, 4), "type2_auc": round(type2_auc, 4),
                   "d_star": round(d_star, 4), "mc_raw": round(mc, 4),
                   "mc_normalized": normalized, "indeterminate": False,
                   "n_parse_failures": n_parse_failures,
                   "n_signal_trials": n_signal, "n_noise_trials": n_noise}
    with open("/output/t11_sdt_summary.json", "w") as f:
        json.dump(sdt_summary, f, indent=2)

    return round(normalized * completion_rate, 4)


mc_binary_pairs.run(kbench.llm)

%choose t11-msv_mc_binary_pairs


Task 11 - MC Binary Pairs: 80 questions (160 total prompts)


  [prompt failure] TypeError: 'NoneType' object is not subscriptable
  Prompt failure (see error above) — returning partial results
  Valid trials: 141/141 | Parse failures: 0 | Completion: 141/160 (88%)
  Hit rate: 0.688 | FA rate: 0.134 | d-hat: 1.597
  Type-2 AUC: 0.495 | d*: -0.016 | MC: -0.010 (inefficient) | Normalized: 0.495
Kept: t11-msv_mc_binary_pairs-run_id_Run_1_qwen_qwen3-next-80b-a3b-thinking.run.json
Kept: t11-msv_mc_binary_pairs.task.json
